# Fracture Analysis using Complex Variable Method

This notebook demonstrates fracture mechanics analysis using complex variable approaches, specifically applying Muskhelishvili's complex potential theory. The complex variable method is a powerful technique in fracture mechanics for analyzing stress fields around cracks.

## Theoretical Background

In the complex variable approach to elasticity, the stress components are related to two analytic functions $\Phi(z)$ and $\Psi(z)$ of the complex variable $z = x + iy$.

### Key Formulas

The stress components in Cartesian coordinates are given by:

$$\sigma_{xx} + \sigma_{yy} = 4 \text{Re}[\Phi'(z)]$$

$$\sigma_{yy} - \sigma_{xx} + 2i\tau_{xy} = 2[\bar{z}\Phi''(z) + \Psi'(z)]$$

For a crack of length $2a$ along the x-axis, centered at the origin, under Mode I loading (tension), the complex potentials are:

$$\Phi(z) = \frac{\sigma_\infty}{2}\left(z - \frac{a^2}{z}\right)$$

$$\Psi(z) = \frac{\sigma_\infty}{2}\left(z + \frac{a^2}{z} - \frac{2a^2}{z}\right)$$

The stress intensity factor for Mode I is given by:

$$K_I = \sigma_\infty \sqrt{\pi a}$$

For mixed-mode loading, the complex potentials and stress intensity factors have different forms which we'll explore in this notebook.

## Setup and Import Libraries

In [1]:
##%% Import necessary libraries
import numpy as np
import matplotlib.pyplot as plt

from matplotlib import cm
import sympy as sp

from __init__ import *

# Enable matplotlib inline display
%matplotlib inline
plt.rcParams['figure.figsize'] = [10, 8]
plt.rcParams['font.size'] = 12

NameError: name 'plt' is not defined

## Define Complex Potentials for Fracture Analysis

In [ ]:
##%% Define complex potentials for Mode I (tension) crack

def phi_mode1(z, a, sigma_inf):
    """
    First complex potential for Mode I crack
    
    Parameters:
    z : complex - Position in complex plane
    a : float - Half-length of crack
    sigma_inf : float - Far-field stress
    """
    return (sigma_inf/2) * (z - a**2/z)

def psi_mode1(z, a, sigma_inf):
    """
    Second complex potential for Mode I crack
    """
    return (sigma_inf/2) * (z + a**2/z - 2*a**2/z)

def phi_prime_mode1(z, a, sigma_inf):
    """
    Derivative of first complex potential
    """
    return (sigma_inf/2) * (1 + a**2/z**2)

def psi_prime_mode1(z, a, sigma_inf):
    """
    Derivative of second complex potential
    """
    return (sigma_inf/2) * (1 - a**2/z**2 + 2*a**2/z**2)

## Calculate Stress Components

In [ ]:
##%% Define functions to calculate stress components

def stress_sum(z, a, sigma_inf):
    """
    Calculate σ_xx + σ_yy
    """
    return 4 * np.real(phi_prime_mode1(z, a, sigma_inf))

def stress_diff(z, a, sigma_inf):
    """
    Calculate complex value representing (σ_yy - σ_xx) + 2i*τ_xy
    """
    z_conj = np.conjugate(z)
    phi_prime_prime = (sigma_inf/2) * (-2 * a**2 / z**3)
    return 2 * (z_conj * phi_prime_prime + psi_prime_mode1(z, a, sigma_inf))

def stress_components(z, a, sigma_inf):
    """
    Calculate all stress components: σ_xx, σ_yy, τ_xy
    """
    sum_stress = stress_sum(z, a, sigma_inf)
    diff_stress_complex = stress_diff(z, a, sigma_inf)
    
    sigma_yy_minus_xx = np.real(diff_stress_complex)
    tau_xy = np.imag(diff_stress_complex)/2
    
    sigma_xx = (sum_stress - sigma_yy_minus_xx)/2
    sigma_yy = (sum_stress + sigma_yy_minus_xx)/2
    
    return sigma_xx, sigma_yy, tau_xy

## Stress Intensity Factor Calculation

In [ ]:
##%% Define functions for stress intensity factors

def stress_intensity_factor_mode1(a, sigma_inf):
    """
    Calculate stress intensity factor for Mode I
    
    Parameters:
    a : Quantity - Half-length of crack with units
    sigma_inf : Quantity - Far-field stress with units
    
    Returns:
    K_I : Quantity - Stress intensity factor with units MPa·√m
    """
    return sigma_inf * np.sqrt(np.pi * a)

## Demonstration with Units (Pint)

In [ ]:
##%% Demonstration with physical units
from pint import UnitRegistry, Quantity as Q_
# Define parameters with units
a = Q_(0.01, "m")  # Half-crack length (1 cm)
sigma_inf = Q_(100, "MPa")  # Far-field stress (100 MPa)

# Calculate stress intensity factor
K_I = stress_intensity_factor_mode1(a, sigma_inf)
print(f"Stress Intensity Factor (K_I): {K_I:.2f}")

# Convert to imperial units
K_I_imperial = K_I.to("ksi * inch**0.5")
print(f"K_I in imperial units: {K_I_imperial:.2f}")

# Calculate critical crack length for a given material
K_Ic = Q_(50, "MPa * m**0.5")  # Example fracture toughness for a steel
sigma_critical = K_Ic / np.sqrt(np.pi * a)
print(f"Critical stress for fracture: {sigma_critical:.2f}")

## Visualization of Stress Fields

In [ ]:
##%% Create grid for stress field visualization

def create_stress_field_grid(a_value, sigma_value, grid_size=100, x_range=5, y_range=5):
    """
    Create and calculate stress field on a grid
    """
    # Extract numeric values for calculation
    a_num = a_value.to("m").magnitude
    sigma_num = sigma_value.to("Pa").magnitude
    
    # Create grid
    x = np.linspace(-x_range * a_num, x_range * a_num, grid_size)
    y = np.linspace(-y_range * a_num, y_range * a_num, grid_size)
    X, Y = np.meshgrid(x, y)
    Z = X + 1j*Y
    
    # Initialize stress arrays
    sigma_xx = np.zeros_like(X)
    sigma_yy = np.zeros_like(X)
    tau_xy = np.zeros_like(X)
    
    # Calculate stress at each point
    for i in range(grid_size):
        for j in range(grid_size):
            # Skip points very close to or on the crack
            z = Z[i, j]
            if abs(z.imag) < 1e-10 and abs(z.real) < a_num:
                continue
                
            sigma_xx[i, j], sigma_yy[i, j], tau_xy[i, j] = stress_components(z, a_num, sigma_num)
    
    # Add units back
    sigma_xx_with_units = sigma_xx * ureg.Pa
    sigma_yy_with_units = sigma_yy * ureg.Pa
    tau_xy_with_units = tau_xy * ureg.Pa
    
    return X, Y, sigma_xx_with_units, sigma_yy_with_units, tau_xy_with_units

In [ ]:
##%% Plot stress fields

X, Y, sigma_xx, sigma_yy, tau_xy = create_stress_field_grid(a, sigma_inf, grid_size=100)

# Convert to MPa for visualization
sigma_xx_MPa = sigma_xx.to("MPa").magnitude
sigma_yy_MPa = sigma_yy.to("MPa").magnitude
tau_xy_MPa = tau_xy.to("MPa").magnitude

# Von Mises stress
von_mises = np.sqrt(sigma_xx_MPa**2 + sigma_yy_MPa**2 - sigma_xx_MPa*sigma_yy_MPa + 3*tau_xy_MPa**2)

# Create figure with multiple subplots
fig, axes = plt.subplots(2, 2, figsize=(16, 14))

# Plot σ_xx
im1 = axes[0, 0].contourf(X, Y, sigma_xx_MPa, 50, cmap=cm.jet)
axes[0, 0].set_title(r'$\sigma_{xx}$ (MPa)')
axes[0, 0].axis('equal')
plt.colorbar(im1, ax=axes[0, 0])

# Plot σ_yy
im2 = axes[0, 1].contourf(X, Y, sigma_yy_MPa, 50, cmap=cm.jet)
axes[0, 1].set_title(r'$\sigma_{yy}$ (MPa)')
axes[0, 1].axis('equal')
plt.colorbar(im2, ax=axes[0, 1])

# Plot τ_xy
im3 = axes[1, 0].contourf(X, Y, tau_xy_MPa, 50, cmap=cm.jet)
axes[1, 0].set_title(r'$\tau_{xy}$ (MPa)')
axes[1, 0].axis('equal')
plt.colorbar(im3, ax=axes[1, 0])

# Plot von Mises stress
im4 = axes[1, 1].contourf(X, Y, von_mises, 50, cmap=cm.jet)
axes[1, 1].set_title('Von Mises Stress (MPa)')
axes[1, 1].axis('equal')
plt.colorbar(im4, ax=axes[1, 1])

# Add a crack representation in all plots
a_value = a.to("m").magnitude
for ax in axes.flat:
    ax.plot([-a_value, a_value], [0, 0], 'k-', linewidth=3)
    ax.set_xlabel('x (m)')
    ax.set_ylabel('y (m)')

plt.tight_layout()
plt.show()

## Parametric Studies

In [ ]:
##%% Study of stress intensity factor vs crack length

# Range of crack lengths
a_values = np.linspace(0.001, 0.05, 20) * ureg.m
K_I_values = [stress_intensity_factor_mode1(a_val, sigma_inf).magnitude for a_val in a_values]

plt.figure(figsize=(10, 6))
plt.plot(a_values.magnitude * 1000, K_I_values, 'o-', linewidth=2)
plt.xlabel('Crack Half-Length (mm)')
plt.ylabel(r'Stress Intensity Factor $K_I$ (MPa$\sqrt{m}$)')
plt.title('Effect of Crack Length on Stress Intensity Factor')
plt.grid(True)
plt.show()

In [ ]:
##%% Study of stress ahead of crack tip

a_num = a.to("m").magnitude
sigma_num = sigma_inf.to("Pa").magnitude
K_I_num = K_I.to("Pa*m**0.5").magnitude

# Points ahead of crack tip (θ = 0)
r_values = np.logspace(-5, -2, 50)  # from 10 μm to 1 cm from crack tip
x_values = a_num + r_values  # x position ahead of crack tip
y_values = np.zeros_like(r_values)  # y = 0

# Analytical solution for stress ahead of crack tip
sigma_yy_analytical = K_I_num / np.sqrt(2*np.pi*r_values)

# Complex variable solution
z_values = x_values + 1j*y_values
sigma_xx_complex = np.zeros_like(r_values)
sigma_yy_complex = np.zeros_like(r_values)
tau_xy_complex = np.zeros_like(r_values)

for i, z in enumerate(z_values):
    sigma_xx_complex[i], sigma_yy_complex[i], tau_xy_complex[i] = stress_components(z, a_num, sigma_num)

plt.figure(figsize=(10, 6))
plt.loglog(r_values, sigma_yy_analytical/1e6, 'b-', linewidth=2, label='Analytical solution')
plt.loglog(r_values, sigma_yy_complex/1e6, 'ro', markersize=4, label='Complex variable method')
plt.grid(True, which="both", ls="-")
plt.xlabel('Distance from crack tip r (m)')
plt.ylabel(r'$\sigma_{yy}$ (MPa)')
plt.title('Stress Field Ahead of Crack Tip')
plt.legend()
plt.show()

## Extension to Mixed-Mode Problems

In [ ]:
##%% Complex potentials for mixed-mode loading

def phi_mixed_mode(z, a, sigma_inf, tau_inf):
    """
    First complex potential for mixed-mode (I+II) crack
    """
    return (sigma_inf/2) * (z - a**2/z) + 1j * (tau_inf/2) * (z - a**2/z)

def psi_mixed_mode(z, a, sigma_inf, tau_inf):
    """
    Second complex potential for mixed-mode crack
    """
    return (sigma_inf/2) * (z + a**2/z - 2*a**2/z) - 1j * (tau_inf/2) * (z + a**2/z)

def mixed_mode_SIF(a, sigma_inf, tau_inf):
    """
    Calculate stress intensity factors for mixed-mode loading
    """
    K_I = sigma_inf * np.sqrt(np.pi * a)
    K_II = tau_inf * np.sqrt(np.pi * a)
    return K_I, K_II

In [ ]:
##%% Demonstration of mixed-mode loading

# Define shear stress
tau_inf = Q_(50, "MPa")  # Apply shear stress

# Calculate stress intensity factors
K_I, K_II = mixed_mode_SIF(a, sigma_inf, tau_inf)
print(f"Mode I SIF: {K_I:.2f}")
print(f"Mode II SIF: {K_II:.2f}")

# Calculate equivalent stress intensity factor
K_eq = np.sqrt(K_I**2 + K_II**2)
print(f"Equivalent SIF: {K_eq:.2f}")

# Calculate crack propagation angle
theta_c = 2 * np.arctan2(K_I - np.sqrt(K_I**2 + 8 * K_II**2), 4 * K_II)
print(f"Crack propagation angle: {np.degrees(theta_c):.2f} degrees")

## Symbolic Derivation of Key Formulas

In [ ]:
##%% Symbolic derivation of complex variable formulas

# Define symbolic variables
z, zbar, a, sigma = sp.symbols('z zbar a sigma')

# Define complex potentials symbolically
phi_sym = sigma/2 * (z - a**2/z)
psi_sym = sigma/2 * (z + a**2/z - 2*a**2/z)

# Take derivatives
phi_prime_sym = sp.diff(phi_sym, z)
phi_prime_prime_sym = sp.diff(phi_prime_sym, z)
psi_prime_sym = sp.diff(psi_sym, z)

# Stress components formulas
sigma_sum_sym = 4 * sp.re(phi_prime_sym)
sigma_diff_complex_sym = 2 * (zbar * phi_prime_prime_sym + psi_prime_sym)

# Print symbolic expressions
print("Phi'(z) = ", phi_prime_sym)
print("Phi''(z) = ", phi_prime_prime_sym)
print("Psi'(z) = ", psi_prime_sym)
print("\nσ_xx + σ_yy = ", sigma_sum_sym)
print("σ_yy - σ_xx + 2i*τ_xy = ", sigma_diff_complex_sym)

## J-integral Calculation

In [ ]:
##%% J-integral calculation for linear elastic fracture mechanics

def J_integral_LEFM(K_I, K_II, E, nu):
    """
    Calculate J-integral for linear elastic material
    
    Parameters:
    K_I, K_II : Quantity - Stress intensity factors
    E : Quantity - Young's modulus
    nu : float - Poisson's ratio
    """
    # For plane stress
    J_plane_stress = (K_I**2 + K_II**2) / E
    
    # For plane strain
    J_plane_strain = (K_I**2 + K_II**2) * (1 - nu**2) / E
    
    return J_plane_stress, J_plane_strain

# Calculate J-integral for our example
E = Q_(200, "GPa")  # Young's modulus (typical for steel)
nu = 0.3  # Poisson's ratio

J_stress, J_strain = J_integral_LEFM(K_I, K_II, E, nu)
print(f"J-integral (plane stress): {J_stress.to('kJ/m^2'):.4f}")
print(f"J-integral (plane strain): {J_strain.to('kJ/m^2'):.4f}")

## Conclusion

In this notebook, we've demonstrated the application of complex variable methods to fracture mechanics problems. The Muskhelishvili approach using complex potentials is particularly powerful for calculating stress fields around cracks and determining stress intensity factors.

Key accomplishments:
1. Implemented complex potentials for Mode I and mixed-mode loading
2. Calculated and visualized stress fields around cracks
3. Determined stress intensity factors and demonstrated their relationship with crack length
4. Computed the J-integral for energy release rate assessment
5. Verified the singular stress field ahead of the crack tip

This framework can be extended to more complex geometries and loading conditions by modifying the complex potentials accordingly.